# Experiment 5: what happens after a cold front?

**Part 1, EDA:** how do the cold fronts in our data look in all variables? Are there tells before the drop, and
how long does the cold air stay?

**Part 2, forecasts after the front:** if the forecast starts *after* the drop, does TiRex-2 expect the cold
to last as long as it really does? Is it less sure (wider band) than on a normal day?

Fronts: the 16 accepted cold fronts of the team labels (PR #11), every passage at every point. As in
Experiment 3, the "drop start" of a passage is the steepest unusual 3 h cooling within ±12 h of the label.

**"Unusual" (anomaly):** a value minus the usual value of that point at that month and hour of day (the
average over the year). This removes the normal day-night cycle, so 0 = usual for that time.

In [1]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from plotly.subplots import make_subplots
from tirex2 import TimeseriesType, load_model

DATA = Path("/workspace/data/austria_15km")
FIG = Path("/workspace/phase1-model-behavior/pair-1/experiments/figures/exp5")
FIG.mkdir(parents=True, exist_ok=True)
EXP3 = Path("/workspace/data/notebooks/results/exp3_forecasts_30perfront.npz")
RESULTS = Path("/workspace/data/notebooks/results/exp5_after_front.npz")
C, H = 10 * 168, 24     # history (10 weeks, as in Experiment 3), forecast hours
OFFSETS = [6, 12, 24, 48]  # forecast start, hours after the drop start
RERUN = False
QUICK = bool(os.environ.get("QUICK"))
if QUICK:  # quick code test
    FIG = Path("/tmp")
    RESULTS = Path("/tmp/exp5_after_front.npz")

In [2]:
VARS = ["T2M", "TD2M", "P0", "RH2M", "UU", "VV", "RR"]
df = pd.read_parquet(DATA / "grid_long.parquet", columns=["point_id", "time", *VARS])
wide = {v: df.pivot(index="time", columns="point_id", values=v).interpolate(limit=3, limit_area="inside") for v in VARS}
times, pts = wide["T2M"].index, wide["T2M"].columns
col = {p: i for i, p in enumerate(pts)}
X = {v: w.to_numpy(np.float32) for v, w in wide.items()}
X["P0"] = X["P0"] / 100  # Pa -> hPa
X["WS"] = np.hypot(X["UU"], X["VV"])
T = X["T2M"]
usual = {v: pd.DataFrame(a).groupby([times.month, times.hour]).transform("mean").to_numpy() for v, a in X.items()}
anom = {v: X[v] - usual[v] for v in X}

fronts = pd.read_csv(DATA / "fronts.csv", parse_dates=["peak_time"])
fronts = fronts[fronts.cold_front].reset_index(drop=True)
ev = pd.read_parquet(DATA / "events.parquet")
onsets = (ev[ev.kind == "cold_front"].groupby("episode")
          .agg(point_id=("point_id", "first"), onset=("time", "min")).reset_index(drop=True))
d3 = np.full_like(T, np.nan)
d3[:-3] = T[3:] - T[:-3]
d3a = d3 - pd.DataFrame(d3).groupby([times.month, times.hour]).transform("mean").to_numpy()
rows = []
for _, f in fronts.iterrows():
    lo, hi = f.peak_time - pd.Timedelta("12h"), f.peak_time + pd.Timedelta("12h")
    on = onsets[(onsets.onset >= lo) & (onsets.onset < hi)].groupby("point_id").onset.min()
    for p, t in on.items():
        i = times.get_loc(t)
        w = np.arange(max(i - 12, 0), min(i + 13, len(times) - 3))
        rows.append({"front": f.peak_time, "point_id": p, "j": int(w[np.nanargmin(d3a[w, col[p]])])})
pas = pd.DataFrame(rows)
print(f"{len(fronts)} fronts, {len(pas)} passages")

16 fronts, 4992 passages


## Part 1: the average cold front in all variables

All passages aligned on the drop start (hour 0). Line = median over all passages, band = middle half (25-75 %).
Values are anomalies (0 = usual for that point, month and hour), except rain (mean, mm per hour).
Dotted line at −6 h = where the forecasts of Experiment 3 started.

In [3]:
REL = np.arange(-48, 97)
PANELS = [("T2M", "temperature (°C)"), ("TD2M", "dew point (°C)"), ("P0", "pressure (hPa)"), ("RH2M", "humidity (%)"),
          ("WS", "wind speed (m/s)"), ("UU", "wind from west (m/s)"), ("VV", "wind from south (m/s)"), ("RR", "rain (mm/h)")]
keep = pas[(pas.j + REL[0] >= 0) & (pas.j + REL[-1] < len(times))]
comp = {v: np.stack([(X if v == "RR" else anom)[v][r.j + REL, col[r.point_id]] for r in keep.itertuples()]) for v, _ in PANELS}

fig = make_subplots(rows=4, cols=2, shared_xaxes=True, vertical_spacing=0.06, horizontal_spacing=0.08,
                    subplot_titles=[t for _, t in PANELS])
for n, (v, title) in enumerate(PANELS):
    r, c = n // 2 + 1, n % 2 + 1
    a = comp[v]
    if v == "RR":
        fig.add_trace(go.Scatter(x=REL, y=np.nanmean(a, 0), mode="lines", line={"color": "#1f77b4", "width": 2.2}, showlegend=False,
                                 hovertemplate="%{x:+d} h: %{y:.2f}<extra>rain (mean)</extra>"), row=r, col=c)
    else:
        q25, q50, q75 = np.nanpercentile(a, [25, 50, 75], axis=0)
        fig.add_trace(go.Scatter(x=np.r_[REL, REL[::-1]], y=np.r_[q75, q25[::-1]], mode="lines", fill="toself", line={"width": 0},
                                 fillcolor="rgba(31,119,180,0.18)", hoverinfo="skip", showlegend=False), row=r, col=c)
        fig.add_trace(go.Scatter(x=REL, y=q50, mode="lines", line={"color": "#1f77b4", "width": 2.2}, showlegend=False,
                                 hovertemplate="%{x:+d} h: %{y:+.1f}<extra>" + title + "</extra>"), row=r, col=c)
        fig.add_hline(y=0, line={"color": "#999", "width": 1}, row=r, col=c)
    fig.add_vline(x=0, line={"color": "white", "width": 1}, row=r, col=c)
    fig.add_vline(x=-6, line={"color": "grey", "width": 1, "dash": "dot"}, row=r, col=c)
fig.update_xaxes(dtick=24, row=4)
fig.update_xaxes(title_text="hours from the drop start", row=4)
fig.update_layout(height=900, template="plotly_dark",
                  title=f"The average cold front in all variables ({len(keep)} passages, {keep.front.nunique()} fronts)"
                        "<br><sup>anomaly = difference from the usual value at that point, month and hour; band = middle half of the passages</sup>")
fig.write_html(FIG / "exp5_average_front_all_variables.html", include_plotlyjs="cdn")
fig.show()

## Part 1: how long does the cold stay?

For every passage: the drop = usual-adjusted temperature before the front (mean of the 6 h before the drop start)
minus its lowest value in the 12 h after. **Duration** = hours from the drop start until half of that drop is made
up again. Only passages with a drop of at least 2 °C; passages not half recovered within 7 days are shown as "> 7 d".

In [4]:
MAXH = 168
dur_rows = []
a = anom["T2M"]
for r in pas.itertuples():
    c = col[r.point_id]
    if r.j - 6 < 0 or r.j + 12 + MAXH >= len(times):
        continue
    pre = np.nanmean(a[r.j - 6 : r.j, c])
    m = r.j + int(np.nanargmin(a[r.j : r.j + 13, c]))
    drop = pre - a[m, c]
    if not drop >= 2:
        continue
    k = np.where(a[m : r.j + MAXH, c] >= a[m, c] + drop / 2)[0]
    dur_rows.append({"front": r.front, "point_id": r.point_id, "drop": drop, "duration": (m + k[0] - r.j) if len(k) else np.inf})
dur = pd.DataFrame(dur_rows)
fin = np.isfinite(dur.duration)
print(f"{len(dur)} passages with a drop of at least 2 °C (median drop {dur['drop'].median():.1f} °C)")
print(f"half recovered after: median {dur.duration.median():.0f} h, middle half {dur.duration.quantile(.25):.0f}-{dur.duration.quantile(.75):.0f} h, "
      f"not within 7 days: {(~fin).mean():.0%}")

fig = make_subplots(rows=1, cols=2, column_widths=[0.45, 0.55], horizontal_spacing=0.12,
                    subplot_titles=["All passages", "Per front (one dot per point, diamond = median)"])
fig.add_trace(go.Histogram(x=np.minimum(dur.duration, MAXH + 6), xbins={"start": 0, "end": MAXH + 12, "size": 6},
                           marker_color="#1f77b4", showlegend=False, hovertemplate="%{x} h: %{y} passages<extra></extra>"), row=1, col=1)
order = dur.groupby("front").duration.median().sort_values().index
for f in order:
    g = dur[dur.front == f]
    y = f"{f:%Y-%m-%d}"
    fig.add_trace(go.Scatter(x=np.minimum(g.duration, MAXH + 6), y=[y] * len(g), mode="markers", showlegend=False,
                             marker={"color": "rgba(31,119,180,0.35)", "size": 5}, hovertemplate="%{x:.0f} h<extra>" + y + "</extra>"), row=1, col=2)
    fig.add_trace(go.Scatter(x=[min(g.duration.median(), MAXH + 6)], y=[y], mode="markers", showlegend=False,
                             marker={"color": "white", "symbol": "diamond", "size": 10},
                             hovertemplate="median %{x:.0f} h<extra>" + y + "</extra>"), row=1, col=2)
tick = {"tickvals": [0, 24, 48, 72, 96, 120, 144, MAXH + 6], "ticktext": ["0", "1 d", "2 d", "3 d", "4 d", "5 d", "6 d", "> 7 d"]}
fig.update_xaxes(title_text="hours until half recovered", **tick)
fig.update_yaxes(title_text="passages", row=1, col=1)
fig.update_layout(height=520, template="plotly_dark",
                  title=f"How long does the cold stay after a front? ({len(dur)} passages with a drop ≥ 2 °C)")
fig.write_html(FIG / "exp5_cold_duration.html", include_plotlyjs="cdn")
fig.show()

4743 passages with a drop of at least 2 °C (median drop 8.2 °C)
half recovered after: median 17 h, middle half 12-35 h, not within 7 days: 5%


## Part 2: forecasts that start after the drop

Same cases as Experiment 3 (15 fronts × 30 points, only those with a real drop of at least 2 °C). Temperature
only, 10 weeks of history. The forecast starts 6, 12, 24 or 48 h after the drop start.

**Normal-day reference:** the same point and hour of day, 2 or 3 weeks before or after the front, at a time with
no front passage at that point within 2 days.

**Measures** (over the 24 forecast hours)
- **band width (°C):** width of the 10-90 % band, i.e. how unsure the model is
- **inside the band (%):** share of hours where what happened is inside the band (80 % would be perfect)
- **too warm (+) / too cold (−) (°C):** forecast median minus what happened

In [5]:
z = np.load(EXP3)
real_drop = z["truth"][:, 6] - z["truth"][:, 6:13].min(axis=1)
cases = pd.DataFrame({"point_id": z["point_id"], "front": pd.to_datetime(z["front"]),
                      "j": [times.get_loc(pd.Timestamp(s)) + 6 for s in z["start"]]})[real_drop >= 2]
cases = cases[cases.j + max(OFFSETS) + H <= len(times)].reset_index(drop=True)

passage_hours = {p: g.j.to_numpy() for p, g in pas.groupby("point_id")}


def normal_start(p, j):
    for d in (-14, -21, 14, 21):
        s = j + 24 * d
        if s - C < 0 or s + H > len(times):
            continue
        if not np.any(np.abs(passage_hours[p] - s) < 48 + H):
            return s
    return -1


cases["normal"] = [normal_start(p, j) for p, j in zip(cases.point_id, cases.j)]
if QUICK:
    cases = cases.iloc[:8].reset_index(drop=True)
print(f"{len(cases)} cases, {(cases.normal >= 0).sum()} with a normal-day reference")


def truth_at(starts, pids):
    return np.stack([T[s : s + H, col[p]] for p, s in zip(pids, starts)])


model = load_model("NX-AI/TiRex-2", device="cpu")


def forecast(starts, pids):
    ts = [TimeseriesType(torch.from_numpy(T[s - C : s, col[p]][None].copy()), None, None) for p, s in zip(pids, starts)]
    out = model.forecast(ts, prediction_length=H, output_type="numpy", batch_size=64)
    return np.stack([q[0] for q in out])


norm = cases[cases.normal >= 0]
starts = {f"+{o} h": (cases.j + o).to_numpy() for o in OFFSETS}
pids = {k: cases.point_id for k in starts}
starts["normal day"], pids["normal day"] = norm.normal.to_numpy(), norm.point_id
fc = {}
if RESULTS.exists() and not RERUN and not QUICK:
    zz = np.load(RESULTS)
    assert list(zz["point_id"]) == list(cases.point_id)
    fc = {k: zz["fc " + k] for k in starts}
    print(f"loaded saved forecasts from {RESULTS.name}")
else:
    for k, value in starts.items():
        t0 = time.time()
        fc[k] = forecast(value, pids[k])
        print(f"{k:12s} {time.time() - t0:4.0f} s")
truth = {k: truth_at(starts[k], pids[k]) for k in starts}
if not RESULTS.exists() or RERUN or QUICK:
    np.savez_compressed(RESULTS, point_id=cases.point_id.to_numpy(), front=cases.front.astype(str).to_numpy().astype("U"),
                        drop_start=np.asarray(times[cases.j].astype(str)).astype("U"), normal_start=np.asarray(
                            [str(times[s]) if s >= 0 else "" for s in cases.normal]).astype("U"),
                        dur_point_id=dur.point_id.to_numpy(), dur_front=dur.front.astype(str).to_numpy().astype("U"),
                        dur_drop=dur["drop"].to_numpy(), dur_hours=dur.duration.to_numpy(), comp_rel=REL,
                        comp_T2M_median=np.nanmedian(comp["T2M"], axis=0),
                        **{"fc " + k: v for k, v in fc.items()}, **{"truth " + k: v for k, v in truth.items()})

KEYS = list(starts)
SHADE = {"+6 h": "#2171b5", "+12 h": "#4292c6", "+24 h": "#6baed6", "+48 h": "#9ecae1", "normal day": "#a0a0a0"}
width = {k: (fc[k][:, 8] - fc[k][:, 0]).mean(axis=1) for k in KEYS}
inside = {k: ((truth[k] >= fc[k][:, 0]) & (truth[k] <= fc[k][:, 8])).mean(axis=1) for k in KEYS}
pd.DataFrame({"band width (°C)": [width[k].mean() for k in KEYS],
              "inside the band (%)": [100 * inside[k].mean() for k in KEYS],
              "error (°C)": [np.abs(fc[k][:, 4] - truth[k]).mean() for k in KEYS],
              "too warm (+) / too cold (−) (°C)": [(fc[k][:, 4] - truth[k]).mean() for k in KEYS]},
             index=pd.Index(KEYS, name="forecast start")).round(2)

269 cases, 241 with a normal-day reference


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

loaded saved forecasts from exp5_after_front.npz


,band width (°C),inside the band (%),error (°C),too warm (+) / too cold (−) (°C)
forecast start,,,,
+6 h,6.27,74.32,2.22,0.64
+12 h,5.96,71.07,2.25,0.51
+24 h,5.55,76.52,1.91,-0.53
+48 h,5.57,81.46,1.76,0.12
normal day,5.81,74.69,2.06,-1.25


### Is the model less sure after a front?

Left: band width per case (one dot each). Right: how often what happened was inside the band; the dashed line is
80 %, what a 10-90 % band should reach.

In [6]:
fig = make_subplots(rows=1, cols=2, column_widths=[0.6, 0.4], horizontal_spacing=0.12,
                    subplot_titles=["Band width (how unsure)", "What happened inside the band"])
for k in KEYS:
    fig.add_trace(go.Box(y=width[k], name=k, marker={"color": SHADE[k], "size": 3}, line={"color": SHADE[k]}, boxpoints="all",
                         jitter=0.4, pointpos=0, showlegend=False, hovertemplate="%{y:.1f} °C<extra>" + k + "</extra>"), row=1, col=1)
    fig.add_trace(go.Bar(x=[k], y=[100 * inside[k].mean()], marker_color=SHADE[k], showlegend=False,
                         hovertemplate="%{y:.0f} %<extra>" + k + "</extra>"), row=1, col=2)
fig.add_hline(y=80, line={"color": "white", "dash": "dash", "width": 1}, row=1, col=2)
fig.update_yaxes(title_text="10-90 % band width (°C)", rangemode="tozero", row=1, col=1)
fig.update_yaxes(title_text="hours inside the band (%)", range=[0, 100], row=1, col=2)
fig.update_xaxes(title_text="forecast start (after the drop start)")
fig.update_layout(height=480, template="plotly_dark",
                  title=f"After a cold front: is the model less sure than on a normal day?<br><sup>{len(cases)} front cases, "
                        f"{len(norm)} normal-day references at the same points; 24 h forecasts, 10 weeks of history</sup>")
fig.write_html(FIG / "exp5_after_front_band.html", include_plotlyjs="cdn")
fig.show()

### Does the model expect the cold to last?

Usual-adjusted temperature (0 = usual for that point and hour), averaged over all cases, aligned on the drop start.
White = what happened over 3 days, colours = the mean forecast from each start. A forecast that rises towards
0 faster than the white line expects the cold to end too early.

In [7]:
SPAN = np.arange(-12, max(OFFSETS) + H)
tru_a = np.stack([anom["T2M"][j + SPAN, col[p]] for p, j in zip(cases.point_id, cases.j)])
fig = go.Figure()
fig.add_hline(y=0, line={"color": "#999", "width": 1}, annotation_text="usual for that hour", annotation_position="top right")
fig.add_vline(x=0, line={"color": "white", "width": 1})
fig.add_trace(go.Scatter(x=SPAN, y=tru_a.mean(0), mode="lines", line={"color": "white", "width": 3.5}, name="what happened",
                         hovertemplate="%{x:+d} h: %{y:+.1f} °C<extra>what happened</extra>"))
for o in OFFSETS:
    k = f"+{o} h"
    hrs = o + np.arange(H)
    us = np.stack([usual["T2M"][j + hrs, col[p]] for p, j in zip(cases.point_id, cases.j)])
    fig.add_trace(go.Scatter(x=hrs, y=(fc[k][:, 4] - us).mean(0), mode="lines", line={"color": SHADE[k], "width": 2.5},
                             name=f"forecast from {k}", hovertemplate="%{x:+d} h: %{y:+.1f} °C<extra>" + k + "</extra>"))
    fig.add_trace(go.Scatter(x=[o], y=[(tru_a[:, np.searchsorted(SPAN, o)]).mean()], mode="markers", showlegend=False,
                             marker={"color": SHADE[k], "size": 9, "line": {"color": "white", "width": 1}}, hoverinfo="skip"))
fig.update_layout(height=470, template="plotly_dark", legend={"orientation": "h", "y": -0.2},
                  title=f"Does the model expect the cold to last? ({len(cases)} cases)<br><sup>dots = forecast starts; "
                        "y = temperature minus the usual value for that point and hour</sup>",
                  xaxis={"title": "hours from the drop start", "dtick": 12}, yaxis={"title": "unusual temperature (°C)"})
fig.write_html(FIG / "exp5_does_cold_last.html", include_plotlyjs="cdn")
fig.show()

## Reusing the results

Saved in `data/notebooks/results/exp5_after_front.npz`:

```python
z = np.load("/workspace/data/notebooks/results/exp5_after_front.npz")
z["point_id"], z["front"], z["drop_start"], z["normal_start"]  # one entry per case ("" = no normal-day reference)
z["fc +6 h"], z["truth +6 h"]       # (cases, 9 quantiles, 24 h) and (cases, 24); also +12 h, +24 h, +48 h
z["fc normal day"], z["truth normal day"]  # only the cases with a normal-day reference
z["dur_point_id"], z["dur_front"], z["dur_drop"], z["dur_hours"]  # duration per passage (inf = not within 7 days)
z["comp_rel"], z["comp_T2M_median"]  # average front shape (unusual temperature), used by Experiment 6
```